In [1]:
import os
import json
import numpy as np
import cv2
import fnv.file
from datetime import datetime, timezone
from tqdm.auto import tqdm

# Tabella di mapping derivata da agent.md
MAPPING = {
    '101': 'Rec-022.seq',
    '301': 'Rec-025.seq',
    '901': 'Rec-024.seq',
    '992': 'Rec-027.seq',
    '602': 'Rec-026.seq',
    # Provino 601: file con nomenclatura diversa dagli altri Rec-0XX
    '601': 'Rec-G3_S60.seq',
    '302': 'Rec-028.seq',
    '102': 'Rec-029.seq',
    '902': 'Rec-030.seq',
    '303': 'Rec-031.seq',
    '304': 'Rec-032.seq'
}

DATI_DIR = "dati"
SEQ_DIR = os.path.join(DATI_DIR, "seq")
OUT_DIR = os.path.join(DATI_DIR, "frame_thermo")

os.makedirs(OUT_DIR, exist_ok=True)

def resolve_seq_path(seq_filename):
    """Restituisce il percorso reale del file .seq, gestendo anche nomi senza estensione
    o differenze maiuscole/minuscole. Utile per file non standard come Rec-G3_S60.
    """
    candidates = [seq_filename]
    base, ext = os.path.splitext(seq_filename)
    if ext.lower() != ".seq":
        candidates.append(seq_filename + ".seq")

    # 1) Prova diretta sui candidati
    for name in candidates:
        path = os.path.join(SEQ_DIR, name)
        if os.path.exists(path):
            return path

    # 2) Fallback case-insensitive nella cartella dati/seq
    if os.path.isdir(SEQ_DIR):
        target_bases = {os.path.splitext(name)[0].lower() for name in candidates}
        target_names = {name.lower() for name in candidates}
        for fname in os.listdir(SEQ_DIR):
            fbase = os.path.splitext(fname)[0].lower()
            if fname.lower() in target_names or fbase in target_bases:
                return os.path.join(SEQ_DIR, fname)

    # 3) Percorso atteso, usato solo per stampare un messaggio chiaro di errore
    return os.path.join(SEQ_DIR, candidates[0])

def extract_timestamp(frame_info):
    for field in frame_info:
        if field['name'] == 'Time':
            date_str = field['value']
            try:
                day_of_year = int(date_str[0:3])
                hour        = int(date_str[4:6])
                minute      = int(date_str[7:9])
                second      = int(date_str[-9:-7])
                microsecond = int(date_str[-6])
                
                year  = datetime.now().year
                month = datetime.strptime(str(day_of_year), '%j').month
                day   = datetime.strptime(str(day_of_year), '%j').day
                
                return datetime(
                    year=year, month=month, day=day,
                    hour=hour, minute=minute, second=second,
                    microsecond=microsecond,
                    tzinfo=timezone.utc
                )
            except Exception:
                return None
    return None

def process_video(provino_id, seq_filename):
    seq_path = resolve_seq_path(seq_filename)
    if not os.path.exists(seq_path):
        print(f"File {seq_path} non trovato. Salto.")
        return
        
    provino_out_dir = os.path.join(OUT_DIR, provino_id)
    os.makedirs(provino_out_dir, exist_ok=True)
    
    # Controllo di idempotenza (ripristino): se il file dei metadati esiste già,
    # significa che l'estrazione per questo provino è già stata completata con successo.
    metadata_path = os.path.join(provino_out_dir, "thermal_metadata.json")
    if os.path.exists(metadata_path):
        print(f"[INFO] Provino {provino_id} gia completato ed estratto. Salto.")
        return
    
    print(f"Apertura video {seq_filename} per provino {provino_id}...")
    frame_object = fnv.file.ImagerFile(seq_path)
    h, w = frame_object.height, frame_object.width
    num_frames = frame_object.num_frames
    
    # Calcolo min e max per la normalizzazione del singolo video (range termico)
    vmin = float('inf')
    vmax = float('-inf')
    
    # Analisi range con progress bar
    for i in tqdm(range(num_frames), desc=f"Analisi range {provino_id}", leave=False):
        frame_object.get_frame(i)
        frame = np.array(frame_object.original, copy=True).reshape(h, w)
        vmin = min(vmin, frame.min())
        vmax = max(vmax, frame.max())
    
    # Estrazione frame con progress bar (ultra-veloce tramite OpenCV)
    for i in tqdm(range(num_frames), desc=f"Estrazione frame {provino_id}", leave=False):
        frame_object.get_frame(i)
        frame = np.array(frame_object.original, copy=True).reshape(h, w)
        
        normalised = (frame - vmin) / (vmax - vmin) if vmax > vmin else frame
        gray_img = (normalised * 255).astype(np.uint8)
        
        ts = extract_timestamp(frame_object.frame_info)
        if ts is not None:
            # Formattiamo il timestamp in modo pulito
            ts_str = ts.strftime("%Y%m%d_%H%M%S_%f")
            filename = f"{i:05d}_{ts_str}.png"
        else:
            filename = f"{i:05d}.png"
            
        filepath = os.path.join(provino_out_dir, filename)
        cv2.imwrite(filepath, gray_img)
        
    frame_object.close()
    
    # Salvataggio metadati solo al completamento dell'estrazione con successo
    metadata = {
        "provino_id": provino_id,
        "seq_filename": seq_filename,
        "vmin_raw": int(vmin),
        "vmax_raw": int(vmax),
        "vmin_celsius": round((vmin / 100.0) - 273.15, 2),
        "vmax_celsius": round((vmax / 100.0) - 273.15, 2)
    }
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=4)
        
    print(f"[OK] Completato {provino_id}: {num_frames} frame salvati in {provino_out_dir}")

# Esecuzione per tutti i video mappati con barra di progresso esterna
for provino_id, seq_filename in tqdm(MAPPING.items(), desc="Progresso provini"):
    process_video(provino_id, seq_filename)


Progresso provini:   0%|          | 0/11 [00:00<?, ?it/s]

[INFO] Provino 101 gia completato ed estratto. Salto.
[INFO] Provino 301 gia completato ed estratto. Salto.
[INFO] Provino 901 gia completato ed estratto. Salto.
[INFO] Provino 992 gia completato ed estratto. Salto.
[INFO] Provino 602 gia completato ed estratto. Salto.
File dati\seq\Rec-G3_S60.seq non trovato. Salto.
[INFO] Provino 302 gia completato ed estratto. Salto.
[INFO] Provino 102 gia completato ed estratto. Salto.
[INFO] Provino 902 gia completato ed estratto. Salto.
[INFO] Provino 303 gia completato ed estratto. Salto.
[INFO] Provino 304 gia completato ed estratto. Salto.


# Tracciamento dell'Ugello della Stampante 3D (Template Matching / Convoluzione)

In questa sezione implementiamo il tracciamento automatico dell'ugello estrusore nei video termici.

### Metodologia:
- Utilizziamo l'immagine dell'ugello fornita come kernel/template in `dati/kernel.png`.
- Applichiamo la convoluzione spaziale (Template Matching normalizzato via OpenCV `cv2.matchTemplate` con il metodo `cv2.TM_CCOEFF_NORMED`).
- Troviamo il picco di massima correlazione per tracciare la posizione esatta $(x_c, y_c)$ dell'ugello in ogni frame.
- Visualizziamo interattivamente il tracciamento del singolo frame e calcoliamo l'intera traiettoria per ciascun provino.

### 2. Scansione e Tracciamento dell'Intero Video con Salvataggio Traiettoria

In [4]:
# Ciclo di tracciamento automatico per TUTTI i provini rilevati
trajectories = {}

# 1. Tracciamento o caricamento delle traiettorie per tutti i video
for video_id in tqdm(available_videos, desc="Progresso complessivo tracciamento"):
    trajectory_file = os.path.join(DATI_DIR, f"trajectory_{video_id}.json")
    
    # Se il provino è già stato tracciato, carichiamo la traiettoria esistente
    if os.path.exists(trajectory_file):
        print(f"[INFO] Provino {video_id} gia completato. Traiettoria esistente caricata da '{trajectory_file}'.")
        with open(trajectory_file, 'r') as f:
            trajectories[video_id] = json.load(f)
        continue
        
    # Altrimenti procediamo al tracciamento completo
    files = get_frame_files_track(video_id)
    if not files:
        print(f"[WARNING] Nessun frame disponibile per il provino '{video_id}'. Salto.")
        continue
        
    print(f"Avvio tracciamento termico dell'ugello per il provino {video_id} ({len(files)} frame)...")
    trajectory = []
    
    # Tracciamento ad alte prestazioni
    for idx, filename in enumerate(tqdm(files, desc=f"Tracciamento {video_id}", leave=False)):
        filepath = os.path.join(FRAME_DIR, video_id, filename)
        img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
        
        # Template Matching normalizzato (TM_CCOEFF_NORMED) con lo stesso kernel dell'ugello
        res = cv2.matchTemplate(img, kernel, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, max_loc = cv2.minMaxLoc(res)
        
        # Calcolo del centro geometrico dell'ugello
        cx = max_loc[0] + w_k // 2
        cy = max_loc[1] + h_k // 2
        
        trajectory.append({
            "frame_idx": idx,
            "filename": filename,
            "x": int(cx),
            "y": int(cy),
            "score": float(max_val)
        })
        
    # Salvataggio su file JSON per ripristini futuri
    with open(trajectory_file, 'w') as f:
        json.dump(trajectory, f, indent=4)
        
    print(f"[OK] Completato {video_id}! Traiettoria salvata in '{trajectory_file}'.\n")
    trajectories[video_id] = trajectory

# 2. Generazione del mega-plot comparativo di tutti i provini tracciati
if trajectories:
    print("\nGenerazione del report comparativo delle traiettorie...")
    n_vids = len(trajectories)
    cols = 2
    rows = (n_vids + 1) // 2
    
    fig, axs = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axs = axs.flatten()
    
    for i, (video_id, traj) in enumerate(sorted(trajectories.items())):
        x_coords = [p['x'] for p in traj]
        y_coords = [p['y'] for p in traj]
        
        ax = axs[i]
        ax.plot(x_coords, y_coords, color='darkviolet', alpha=0.7, linewidth=1.8, label='Traiettoria Ugello')
        ax.scatter(x_coords[0], y_coords[0], color='limegreen', marker='o', s=80, zorder=5, label='Inizio')
        ax.scatter(x_coords[-1], y_coords[-1], color='crimson', marker='x', s=80, zorder=5, label='Fine')
        ax.set_title(f"Provino {video_id} ({len(traj)} frame)", fontsize=11, fontweight='bold')
        ax.set_xlabel("Coordinata X (pixel)", fontsize=9)
        ax.set_ylabel("Coordinata Y (pixel)", fontsize=9)
        ax.invert_yaxis()  # Coordinate OpenCV
        ax.grid(True, linestyle='--', alpha=0.4)
        ax.legend(loc='best', fontsize=8)
        
    # Rimuovi eventuali sotto-grafici vuoti se dispari
    for j in range(i + 1, len(axs)):
        fig.delaxes(axs[j])
        
    plt.suptitle("REPORT COMPARATIVO DELLE TRAIETTORIE 2D DELL'UGELLO", fontsize=15, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()
else:
    print("[WARNING] Nessuna traiettoria caricata o elaborata.")

Progresso complessivo tracciamento:   0%|          | 0/10 [00:00<?, ?it/s]

Avvio tracciamento termico dell'ugello per il provino 101 (10343 frame)...


Tracciamento 101:   0%|          | 0/10343 [00:00<?, ?it/s]

KeyboardInterrupt: 